# 06a · 랜덤 워크에서 브라운 운동으로 (From Random Walk to Brownian Motion)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Lawler 2e §8.1 (BM 의 정의, 랜덤 워크 극한) · Ross 12e §10.1 (스케일된 랜덤 워크의 극한으로 BM 정의) · Shreve II §3.2 (scaled random walk), §3.3 (BM 정의와 분포) · G&S 4e §8.5 (Wiener 과정 소개), §13.1–13.2 (BM) · Karlin & Taylor 2e §7.1 (배경) |
| 선수 노트북 | 00a (랜덤 워크·패리티 함정), 00b (MC 오차), 05a (마팅게일·독립 증분), 05d (마팅게일 수렴) |
| 예상 소요 | 90분 |
| 상태 | draft |

00a 에서 동전던지기의 합 $S_n$ 을 만들었고, 05 모듈에서 그것이 마팅게일임을 보았다. 이 노트북은 그 걸음을 **시간은 $n$ 배 촘촘하게, 높이는 $\sqrt n$ 배 작게** 스케일하면 무엇이 남는지 묻는다. 남는 것이 브라운 운동(Brownian motion, BM)이다: 출발점 0, 독립 증분, 가우스 증분 $N(0,t-s)$, 연속 경로. 이 네 조건에서 공분산 $\min(s,t)$ 와 자기유사성이 나오고, 자기유사성은 곧 "어느 배율로 확대해도 똑같이 삐죽삐죽하다 — 접선이 없다" 는 뜻이다. 그림 여섯 장과 닫힌 형식 일곱 개로 이 문장을 확인한다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import norm, binom
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import random_walk_paths
from spkit.brownian import brownian_paths, scaled_random_walk, coarsen
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("06a")
setup_plots()
N_PATHS = scale(20_000)          # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기 (2000)
N_PATHS_DQ = scale(2_000)        # 차분몫 실험용 경로 수 (FAST: 200; 증분은 경로당 1000 개)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_PATHS_DQ={N_PATHS_DQ}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (05d) 마팅게일 수렴 정리의 조건과 결론은 무엇인가?

<details><summary>정답</summary>

$\sup_n E|M_n|<\infty$ ($L^1$ 유계)이면 $M_n\to M_\infty$ a.s. 이고 극한은 유한하다. $L^1$ 수렴까지 얻으려면 균등적분가능성(uniform integrability)이 추가로 필요하다.

</details>

**Q2.** (05d) 처음 빨강 $a$개, 파랑 $b$개인 폴리아 항아리에서 빨강 비율 $X_n$ 의 극한 분포는?

<details><summary>정답</summary>

$X_n$ 은 $[0,1]$ 유계 마팅게일이라 a.s. 수렴하고, 극한은 $\mathrm{Beta}(a,b)$ 다.

</details>

**Q3.** (05d) 골턴-왓슨 과정에서 $Z_n/m^n$ 이 마팅게일인 이유와 극한 $W$ 가 0 이 아닐 조건은?

<details><summary>정답</summary>

$E[Z_{n+1}\mid\mathcal F_n]=mZ_n$ 이므로 $E[Z_{n+1}/m^{n+1}\mid\mathcal F_n]=Z_n/m^n$. $m>1$ 이고 $E[X\log X]<\infty$ (Kesten–Stigum) 이면 $P(W>0)=P(\text{생존})$.

</details>

**Q4.** (05c) 월드 항등식(Wald's identity)을 조건과 함께 써라.

<details><summary>정답</summary>

$X_i$ i.i.d., $E|X_1|<\infty$, 정지시간 $T$ 가 $E[T]<\infty$ 이면 $E[S_T]=E[T]\,E[X_1]$.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **스케일된 랜덤 워크**(scaled random walk): $X_i=\pm1$ 등확률 i.i.d., $S_k=\sum_{i\le k}X_i$,
  $$W^{(n)}_t=\frac{S_{\lfloor nt\rfloor}}{\sqrt n},\qquad t\in[0,1]$$
  (격자점 사이는 계단 또는 선형 보간). $E[W^{(n)}_t]=0$, $\mathrm{Var}(W^{(n)}_t)=\lfloor nt\rfloor/n\to t$. 시간을 $n$ 배 촘촘하게 쓰면 높이는 $\sqrt n$ 으로 나눠야 분산이 $t$ 에 고정된다 — 이 $\sqrt n$ 이 노트북 전체의 열쇠다 (Shreve II §3.2).
- **표준 브라운 운동**(standard Brownian motion) $(B_t)_{t\ge0}$ (Lawler 2e §8.1, Shreve II §3.3):
  (i) $B_0=0$;
  (ii) **독립 증분** — 서로 겹치지 않는 구간들의 증분 $B_{t_2}-B_{t_1},\dots,B_{t_k}-B_{t_{k-1}}$ ($t_1<t_2<\dots<t_k$) 은 독립;
  (iii) **정상 가우스 증분** — $B_t-B_s\sim N(0,t-s)$ ($s<t$);
  (iv) 경로 $t\mapsto B_t$ 는 a.s. 연속.
- **자기유사성**(self-similarity, scaling): 임의의 $c>0$ 에 대해 $(c^{-1/2}B_{ct})_{t\ge0}$ 도 표준 BM. 특히 $B_{ct}\overset d=\sqrt c\,B_t$.
- **여과와 (단순) 마르코프 성질**: $\mathcal F_t=\sigma(B_s:s\le t)$. $(B_{t+s}-B_t)_{s\ge0}$ 는 $\mathcal F_t$ 와 독립인 표준 BM 이다 (정의 (ii)·(iii) 를 다시 읽은 것; 06c 에서 정지시간으로 확장 = 강마르코프).
- **차분몫**(difference quotient) $D_h(t)=(B_{t+h}-B_t)/h\sim N(0,1/h)$: 미분계수의 후보이지만 $h\to0$ 에서 분산 $1/h$ 가 발산한다.

### 2.2 정리 1 — BM 의 기본 성질: 주변분포·공분산·자기유사성 (Shreve II §3.3, G&S 4e §13.2)

$B$ 가 표준 BM 이면
(a) $B_t\sim N(0,t)$;
(b) $\mathrm{Cov}(B_s,B_t)=\min(s,t)$;
(c) $s<t$ 에서 $\rho(B_s,B_t)=\sqrt{s/t}$;
(d) $c>0$ 에 대해 $X_t:=c^{-1/2}B_{ct}$ 는 표준 BM.

가정이 왜 필요한가:
- **독립 증분**: (b) 에서 $E[B_s(B_t-B_s)]=E[B_s]\,E[B_t-B_s]=0$ 을 주는 핵심. 없으면 공분산이 $\min$ 이 아니다 — 예: $X_t=tZ$ ($Z\sim N(0,1)$) 는 $X_t\sim N(0,t^2)$ 이고 공분산이 $st$ 다.
- **증분 분산이 $t-s$ 에 '비례'**: 이것이 $\mathrm{Var}(B_t)=t$ 와 (d) 의 $c^{-1/2}$ 스케일을 결정한다. 분산이 $(t-s)^2$ 이면 경로가 매끄러워지고 스케일은 $c^{-1}$ 이 된다.
- **연속 경로**: (d) 의 $X$ 가 BM 이려면 연속성이 보존되어야 하며, 이는 정의의 일부다. 유한차원 분포만으로는 경로 성질이 결정되지 않는다 (같은 유한차원 분포를 갖고 경로는 불연속인 과정을 만들 수 있다).

<details><summary>증명</summary>

**(a)** 정의 (iii) 에서 $s=0$ 을 넣으면 $B_t-B_0\sim N(0,t)$ 이고 (i) 에 의해 $B_0=0$ 이므로 $B_t\sim N(0,t)$.

**(b)** $s<t$ 라 하자. $B_t=B_s+(B_t-B_s)$ 로 쪼개면 $B_s=B_s-B_0$ 와 $B_t-B_s$ 는 겹치지 않는 구간 $[0,s]$, $[s,t]$ 의 증분이므로 (ii) 에 의해 독립이고, (iii) 에 의해 둘 다 평균 0 이다. 따라서
$$E[B_sB_t]=E[B_s^2]+E[B_s(B_t-B_s)]=s+E[B_s]\,E[B_t-B_s]=s+0=\min(s,t).$$
$E[B_s]=E[B_t]=0$ 이므로 $\mathrm{Cov}(B_s,B_t)=E[B_sB_t]=\min(s,t)$. $s>t$ 는 대칭, $s=t$ 는 (a).

**(c)** $\rho(B_s,B_t)=\dfrac{\mathrm{Cov}(B_s,B_t)}{\sqrt{\mathrm{Var}B_s\,\mathrm{Var}B_t}}=\dfrac{\min(s,t)}{\sqrt{st}}=\dfrac{s}{\sqrt{st}}=\sqrt{s/t}$ ($s<t$).

**(d)** 네 조건을 하나씩 확인한다. (i) $X_0=c^{-1/2}B_0=0$. (iii) $s<t$ 에서 $X_t-X_s=c^{-1/2}(B_{ct}-B_{cs})$ 이고 $B_{ct}-B_{cs}\sim N(0,ct-cs)$ 이므로 $X_t-X_s\sim N(0,c^{-1}(ct-cs))=N(0,t-s)$. (ii) 구간 $[t_{j-1},t_j]$ 들이 겹치지 않으면 $c$ 배 한 구간 $[ct_{j-1},ct_j]$ 들도 겹치지 않으므로 $B$ 의 증분들이 독립이고, 상수배 $c^{-1/2}$ 는 독립성을 바꾸지 않는다. (iv) $t\mapsto ct$ 는 연속이고 $t\mapsto B_t$ 는 a.s. 연속이므로 합성 $t\mapsto B_{ct}$ 도 a.s. 연속이며 상수배도 그렇다. 네 조건이 모두 성립하므로 정의에 의해 $X$ 는 표준 BM 이다. $\square$

</details>

노트북에서는 (b) 를 $E[B_1B_4]=1$, $E[B_1(B_3-B_1)]=0$ 으로, (c) 를 `np.corrcoef` $\approx0.5$ 로, (d) 를 $c=4$ 의 QQ 플롯과 $E[(B_4/2)^2]=1$ 로 확인한다.

### 2.3 정리 2 — Donsker 불변 원리 (함수적 중심극한정리; Lawler 2e §8.1, G&S 4e §13.2)

$EX_1=0$, $\mathrm{Var}X_1=1$ 인 i.i.d. 걸음에 대해 $W^{(n)}\Rightarrow B$ — 경로 공간 $C[0,1]$ 에서 분포 수렴한다. 특히 유한차원 분포 $(W^{(n)}_{t_1},\dots,W^{(n)}_{t_k})\Rightarrow(B_{t_1},\dots,B_{t_k})$ 이고, $\max_{t\le1}W^{(n)}_t\Rightarrow\max_{t\le1}B_t$ 처럼 경로의 연속 범함수도 함께 수렴한다.

가정이 왜 필요한가:
- **평균 0**: 아니면 $W^{(n)}_t\approx\sqrt n\,\mu t+B_t$ 의 드리프트가 $\sqrt n$ 으로 발산한다 (E3(a)).
- **분산 유한**: 무한분산이면 올바른 스케일이 $n^{1/\alpha}$ 이고 극한은 점프를 가진 $\alpha$-안정 과정이 된다 (E3(b)).
- **독립** (또는 약한 종속): 각 구간 증분에 CLT 를 적용하기 위해 필요.

*증명 스케치.* **유한차원.** $t_0=0<t_1<\dots<t_k$ 에 대해 증분 $S_{\lfloor nt_j\rfloor}-S_{\lfloor nt_{j-1}\rfloor}$ 은 서로 다른 걸음들의 합이라 독립이고, 각각 $\lfloor nt_j\rfloor-\lfloor nt_{j-1}\rfloor\sim n(t_j-t_{j-1})$ 개의 걸음의 합이므로 CLT 로 $\sqrt n$ 으로 나눈 것이 $N(0,t_j-t_{j-1})$ 에 수렴한다. 독립 성분의 결합 수렴은 Cramér–Wold 로 얻고, 증분 벡터의 선형변환이 값 벡터이므로 유한차원 분포가 수렴한다. **경로 수준.** 유한차원 수렴에 더해 $\{W^{(n)}\}$ 이 $C[0,1]$ 에서 팽팽함(tightness) — 연속성 모듈러스 $\sup_{|t-s|\le\delta}|W^{(n)}_t-W^{(n)}_s|$ 가 $\delta\to0$ 에서 $n$ 에 대해 균등하게 작다 — 을 보여야 하는데, 이는 최댓값 부등식(Kolmogorov)으로 증명하며 인용한다: Lawler 2e §8.1 의 논의, G&S 4e §13.2, 엄밀 증명은 Billingsley *Convergence of Probability Measures* 2e §8. 노트북에서는 Donsker 그림(4.1)과 한 시점 분포(4.2)로만 확인한다. $\square$

### 2.4 정리 3 — 경로의 미분 불가능성 (Paley–Wiener–Zygmund)

거의 모든 BM 경로는 **모든** $t$ 에서 미분 불가능하다. (약한 형태: 고정된 $t$ 에서 a.s. 미분 불가능.)

가정이 왜 필요한가:
- **증분 분산이 $h$ 에 비례** ($h^2$ 이 아님): 차분몫 $(B_{t+h}-B_t)/h\sim N(0,1/h)$ 의 표준편차가 $h^{-1/2}\to\infty$ 인 유일한 이유. 분산이 $h^2$ 이면 차분몫이 $N(0,1)$ 로 안정되어 미분 가능한 경로가 나온다 ($tZ$ 가 그 예).

*증명 스케치.* **고정 $t$ (완전한 증명).** $t$ 에서 미분 가능하면 어떤 $K<\infty$ 와 $h_0>0$ 이 있어 $0<h<h_0$ 에서 $|B_{t+h}-B_t|\le Kh$ 다. 그런데 $B_{t+h}-B_t=\sqrt h\,Z$, $Z\sim N(0,1)$ 이므로
$$P(|B_{t+h}-B_t|\le Kh)=P(|Z|\le K\sqrt h)\xrightarrow[h\to0]{}0.$$
따라서 임의의 $K$ 에 대해 $P(\text{모든 작은 }h\text{ 에서 }|B_{t+h}-B_t|\le Kh)\le\inf_hP(|Z|\le K\sqrt h)=0$ 이고, $K$ 에 대해 가산 합집합을 취해도 확률 0: 차분몫이 유계일 확률이 0 이므로 a.s. 미분 불가능하다. **모든 $t$ 에서 동시에.** $[0,1]$ 을 $n$ 등분하고, 어떤 $t$ 에서 미분 가능하면 $t$ 근처의 세 개의 연속한 격자 증분이 모두 $\le C/n$ 이어야 함을 이용한다. 독립 증분이므로 그 확률은 $\le P(|Z|\le C n^{-1/2})^3\le(c\,n^{-1/2})^3$ 이고, 이를 $n$ 개 위치에 대해 합하면 $O(n\cdot n^{-3/2})=O(n^{-1/2})\to0$ (Dvoretzky–Erdős–Kakutani 논법; Mörters–Peres *Brownian Motion* Thm 1.30 인용). $\square$

### 2.5 직관

BM 은 **"무한히 잘게 쪼갠 동전던지기의 합"** 이다. 시간 $t$ 동안 $nt$ 번 던지고 $\sqrt n$ 으로 나누면 분산이 $t$ 로 고정된다. 그래서 $B_t$ 의 전형적 크기는 $\sqrt t$ 이고 — $t$ 가 아니다 — 경로는 대부분 포물선 $\pm2\sqrt t$ 안에 머문다. 어느 배율로 확대해도 똑같이 삐죽삐죽하다는 것(자기유사성: 시간 $c$ 배 ↔ 높이 $\sqrt c$ 배)은 확대해도 직선에 가까워지지 않는다, 즉 접선이 없다는 뜻이다. 06b 에서 이 거칠음의 크기(이차 변동 $=t$)를 재고, 07a 에서 그것이 새로운 미적분(이토)을 강제함을 본다.

✍️ 내 말로: '독립 증분'과 '독립 값'의 차이를, $B_1$ 과 $B_4$ 를 예로 들어 두 문장으로 써 보자.

### 2.6 함정 (traps)

- $S_n/\sqrt n$ 의 히스토그램은 $n$ 이 커도 간격 $2/\sqrt n$ 의 **격자** 위에 있다 — 00a 의 패리티 함정 그대로. 정확 확률은 이항으로 계산하고, 정규근사에는 연속성 보정이 필요하다 ($n=400$: 이항 0.1469, 보정 없는 정규 0.1587 — 4.2 절에서 코드로 확인).
- $\mathrm{Var}(B_t)=t$ 이지 $t^2$ 이 아니다. 표준편차는 $\sqrt t$.
- **독립 증분 $\ne$ 독립 값**: $B_1$ 과 $B_4$ 의 상관은 $\sqrt{1/4}=1/2$. 독립인 것은 $B_1$ 과 $B_4-B_1$ 이다.
- "연속이니까 어딘가는 미분 가능하겠지" 는 틀렸다. 차분몫의 표준편차 $h^{-1/2}$ 가 발산한다.
- 시뮬레이션의 BM 은 격자 위 이산 근사다. 격자점 사이의 정보(최댓값, 도달시간)는 잃어버린다 — 06c 의 주제.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — BM 의 네 조건, 정리 1(b) 의 한 줄 증명, 그리고 "왜 $\sqrt n$ 인가" 를 한 문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 손계산 힌트: $S_{400}$ 은 짝수이고 $S_{400}>20\iff S_{400}\ge22$; $B_t\sim N(0,t)$; 정리 1(b)·(c); $D_h\sim N(0,1/h)$.

In [ ]:
predictions = {
    # n=400 걸음 랜덤 워크에서 P(S_400/sqrt(400) > 1)은? (정규근사 1-Phi(1)=0.1587 과 얼마나 다를까? 힌트: 패리티, 이항)
    "p_walk400_gt1": None,
    # P(|B_1| > 1) 은?
    "p_absB1_gt1": None,
    # Var(B_3 - B_1) 은?
    "var_incr_1_3": None,
    # E[B_1 B_4] 는?
    "cov_B1_B4": None,
    # 상관계수 rho(B_1, B_4) 는?
    "corr_B1_B4": None,
    # 창 h = 1e-3 의 차분몫 D_h = (B_{t+h} - B_t)/h 에 대해 E[D_h^2] (= 분산) 은?
    "mean_sq_slope_h": None,
}

## 4. Simulate — 시뮬레이션

이 노트북에서만 쓰는 도우미 세 개를 먼저 정의한다 (각각 두 줄).

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def walk_rescale(S, n):
    """누적 걸음 S (len >= n+1) 의 처음 n 걸음을 Donsker 스케일로: (t_k = k/n, S_k / sqrt(n))."""
    return np.arange(n + 1) / n, S[:n + 1] / np.sqrt(n)

def pareto_steps(size, alpha, rng):
    """대칭 파레토 꼬리 P(|X| > x) = x^(-alpha), x >= 1 (alpha < 2 이면 분산 무한) — E3(b)."""
    return rng.choice([-1, 1], size) * rng.random(size) ** (-1 / alpha)

def qq_plot(x, y, ax, **kw):
    """두 표본의 분위수를 서로 그린다 (길이가 같아야 함) + 대각선."""
    xs, ys = np.sort(x), np.sort(y)
    ax.plot(xs, ys, ".", ms=2, **kw); lim = (min(xs[0], ys[0]), max(xs[-1], ys[-1]))
    ax.plot(lim, lim, "k--", lw=1, label="y = x"); ax.set_aspect("equal")
    return ax

### 4.1 Donsker 그림 — 하나의 긴 걸음을 네 개의 $n$ 으로 (Fig 1)

$\pm1$ 걸음 $S_0,\dots,S_{10000}$ 을 **한 번만** 만들고, $n\in\{10,100,1000,10000\}$ 각각에 대해 처음 $n$ 걸음만 써서 $W^{(n)}_{k/n}=S_k/\sqrt n$ 을 그린다. 네 패널 모두 $t\in[0,1]$, 세로축 $[-3,3]$ 으로 고정한다. 네 패널은 서로 다른 모양이지만 **같은 분포**(Donsker 극한 $B$ 에 가까워지는 분포)를 갖는 경로들이다. 한 가지 더 볼 것: $W^{(10n)}_t=W^{(n)}_{10t}/\sqrt{10}$ ($t\le0.1$) 이므로 각 패널의 왼쪽 $1/10$ (음영) 은 **바로 앞 패널을 가로로 $1/10$, 세로로 $1/\sqrt{10}$ 줄인 것**이다 — 정리 1(d) 의 자기유사성이 걸음 수준에서 이미 보인다.

In [ ]:
N_MAX, N_LIST = 10_000, [10, 100, 1000, 10_000]
S = random_walk_paths(1, N_MAX, rng)[0]                      # S_0 .. S_10000, one walk

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, n in zip(axes.ravel(), N_LIST):
    t_n, W_n = walk_rescale(S, n)
    ax.plot(t_n, W_n, drawstyle="steps-post", lw=0.8 if n >= 1000 else 1.2, color="C0")
    if n > N_LIST[0]:
        ax.axvspan(0, 0.1, color="C1", alpha=0.12)            # = previous panel, shrunk by (1/10, 1/sqrt(10))
    ax.set_title(f"n = {n}:  S_[nt] / sqrt(n)  (first {n} steps of the same walk)")
    ax.set_ylim(-3, 3); ax.set_xlim(0, 1)
for ax in axes[1]: ax.set_xlabel("t")
for ax in axes[:, 0]: ax.set_ylabel("$W^{(n)}_t$")
fig.suptitle("Fig 1. Scaled random walk from ONE walk, n = 10 / 100 / 1000 / 10000 (shaded: previous panel rescaled)")
plt.tight_layout(); plt.show()

$n=10$ 은 계단 열 개, $n=10000$ 은 이미 연속 경로처럼 보인다. 어느 패널도 $\pm3$ 을 거의 넘지 않는다 — $\mathrm{Var}(W^{(n)}_1)\approx1$ 이 $n$ 과 무관하게 유지되기 때문이며, 이것이 $\sqrt n$ 스케일의 역할이다.

### 4.2 한 시점의 분포와 패리티 (Fig 2)

$W^{(400)}_1=S_{400}/20$ 의 히스토그램을 $N(0,1)$ 과 겹친다. **패리티 주의(00a)**: $S_{400}$ 은 짝수만 가능하므로 $S_{400}/20$ 은 간격 $2/\sqrt n=0.1$ 의 격자 위에만 있다. 격자점을 중심으로 하는 폭 0.1 의 bin 을 써야 밀도가 pdf 와 겹친다 (폭 0.05 로 하면 절반이 비어 밀도가 두 배로 보인다). 정확값은 $S_{400}>20\iff S_{400}\ge22\iff(S_{400}+400)/2\ge211$ 에서 이항으로 계산한다.

In [ ]:
N_STEPS_RW = 400
t_rw, W = scaled_random_walk(N_PATHS, N_STEPS_RW, 1.0, rng)   # W[:, k] = S_k / 20
W1 = W[:, -1]
p_walk_hat = mc_proportion(W1 > 1)

p_walk_binom = binom.sf(210, N_STEPS_RW, 0.5)                  # P(S_400 >= 22) = P(Bin(400,1/2) >= 211)
p_walk_normal = 1 - norm.cdf(1.0)                              # uncorrected normal approximation
p_walk_cc = 1 - norm.cdf(21 / 20)                              # continuity-corrected: P(S > 21) under N(0, 400)
print(f"P(S_400/20 > 1): MC {p_walk_hat}")
print(f"  binomial (exact) {p_walk_binom:.6f} | normal, no correction {p_walk_normal:.6f} | continuity-corrected {p_walk_cc:.6f}")
print("lattice check: distinct values of S_400/20 (first 8):", np.unique(np.round(W1, 6))[:8])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.arange(-4.05, 4.15, 0.1)                             # width 0.1 = lattice spacing, centred on lattice points
x = np.linspace(-4, 4, 400)
plot_hist_vs_pdf(W1, x, norm.pdf(x), ax=ax, bins=bins, label_pdf="N(0,1) pdf")
ax.axvline(1, color="C3", ls="--", lw=1,
           label=f"P(>1): binomial {p_walk_binom:.4f}, normal {p_walk_normal:.4f}, MC {p_walk_hat.mean:.4f}")
ax.set_xlabel("$S_{400} / 20$"); ax.set_ylabel("density"); ax.legend(loc="upper left")
ax.set_title(f"Fig 2. One-time distribution of the scaled walk, n = 400 ({N_PATHS} paths, bin width 0.1)")
plt.show()

히스토그램은 $N(0,1)$ 과 겹치지만, $P(>1)$ 은 이항값 0.1469 에 맞고 보정 없는 정규값 0.1587 과는 MC 표준오차의 몇 배나 어긋난다. 이유: 격자 위에서 "$>1$" 은 "$\ge1.1$" 이고, 정규근사에서 이에 해당하는 것은 $>1.05$ (연속성 보정) 이기 때문이다.

### 4.3 BM 의 증분 — 독립·정상성 (Fig 3)

`brownian_paths` 로 $[0,4]$ 위 $dt=0.1$ 격자의 경로를 만들면 열 10, 30, 40 이 $B_1,B_3,B_4$ 다. 정리 1(b) 의 증명이 그대로 코드가 된다: $E[(B_3-B_1)^2]=2$ (정상 증분), $E[B_1(B_3-B_1)]=0$ (독립 증분의 흔적), $E[B_1B_4]=\min(1,4)=1$, $\rho(B_1,B_4)=1/2$.

In [ ]:
t_b, B = brownian_paths(N_PATHS, 40, 4.0, rng)                 # dt = 0.1, columns 10/30/40 = B_1/B_3/B_4
B1, B3, B4 = B[:, 10], B[:, 30], B[:, 40]
assert t_b[10] == 1.0 and t_b[30] == 3.0 and t_b[40] == 4.0

p_absB1_hat = mc_proportion(np.abs(B1) > 1)
var_incr_hat = mc_estimate((B3 - B1) ** 2)
cov_incr_hat = mc_estimate(B1 * (B3 - B1))
cov_B1B4_hat = mc_estimate(B1 * B4)
corr_B1B4_sample = np.corrcoef(B1, B4)[0, 1]
print(f"P(|B_1| > 1)      = {p_absB1_hat}")
print(f"E[(B_3 - B_1)^2]  = {var_incr_hat}   (stationary increments: 3 - 1 = 2)")
print(f"E[B_1 (B_3 - B_1)] = {cov_incr_hat}   (independent increments: 0)")
print(f"E[B_1 B_4]        = {cov_B1B4_hat}   (min(1,4) = 1)")
print(f"corrcoef(B_1, B_4) = {corr_B1B4_sample:.4f}   (sqrt(1/4) = 0.5)")

In [ ]:
n_show = min(2000, N_PATHS)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 5))
axL.scatter(B1[:n_show], (B3 - B1)[:n_show], s=4, alpha=0.4)
axL.set_xlabel("$B_1$"); axL.set_ylabel("$B_3 - B_1$"); axL.set_aspect("equal")
axL.set_title(f"Fig 3a. B_1 vs B_3 - B_1: independent increments\n(round cloud, sample cov = {cov_incr_hat.mean:+.3f})")
axR.scatter(B1[:n_show], B4[:n_show], s=4, alpha=0.4, color="C2")
lim = np.array([-4, 4]); axR.plot(lim, lim, "k--", lw=1, label="y = x  (E[B_4 | B_1] = B_1)")
axR.set_xlabel("$B_1$"); axR.set_ylabel("$B_4$"); axR.set_aspect("equal"); axR.legend(loc="upper left")
axR.set_title(f"Fig 3b. B_1 vs B_4: dependent values\n(corr = {corr_B1B4_sample:.3f}, exact 0.5)")
plt.tight_layout(); plt.show()

왼쪽은 둥근 구름(공분산 0, 세로 폭 $\sqrt2$ 배로 더 큼), 오른쪽은 $y=x$ 방향으로 늘어진 구름이다. $B_4=B_1+(B_4-B_1)$ 에서 $B_1$ 이 그대로 들어 있으므로 $E[B_4\mid B_1]=B_1$ — 마팅게일 성질이 산점도의 기울기 1 로 보인다.

### 4.4 자기유사성 $c^{-1/2}B_{ct}\overset d=B_t$ (Fig 4)

같은 경로에서 $c=4$: $B_4/2$ 의 분위수를 $B_1$ 의 분위수와 QQ 플롯으로 비교하고 $E[(B_4/2)^2]=1$ 을 잰다. 오른쪽에는 경로 5 개를 $[0,4]$ 에 그리고, **같은 경로**를 $(t/4,\,B_t/2)$ 로 변환해 $[0,1]$ 위에 겹친다 — 변환된 경로는 $[0,1]$ 위 표준 BM 과 구별할 수 없다.

In [ ]:
B4_scaled = B4 / 2                                             # c^{-1/2} B_{c·1}, c = 4
selfsim_hat = mc_estimate(B4_scaled ** 2)
print(f"E[(B_4/2)^2] = {selfsim_hat}   (exact 1);  sd(B_4/2) = {B4_scaled.std():.4f}, sd(B_1) = {B1.std():.4f}")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.5))
qq_plot(B1, B4_scaled, axL, label="quantiles")
axL.set_xlabel("quantiles of $B_1$"); axL.set_ylabel("quantiles of $B_4 / 2$"); axL.legend(loc="upper left")
axL.set_title("Fig 4a. QQ plot: B_4/2 vs B_1 (self-similarity, c = 4)")
plot_paths(t_b, B, ax=axR, n_show=5, color="grey", alpha=0.5, lw=1)
plot_paths(t_b / 4, B / 2, ax=axR, n_show=5, lw=1.2)
axR.set_xlabel("t"); axR.set_ylabel("$B_t$  (grey)   |   $B_{4t} / 2$  (colour)")
axR.set_title("Fig 4b. 5 paths on [0,4] (grey) and the same paths as (t/4, B_t/2) on [0,1]")
plt.tight_layout(); plt.show()

QQ 점들이 대각선 위에 있다: $B_4/2$ 와 $B_1$ 은 같은 분포다. 시간을 4 배 압축하면 높이는 $\sqrt4=2$ 배만 줄어든다 — 시간이 $c$ 배면 높이는 $\sqrt c$ 배.

### 4.5 확대해도 거칠다 (Fig 5)

$10^6$ 점 위의 경로 하나를 $[0,1]$, $[0,10^{-2}]$, $[0,10^{-4}]$ 세 창으로 본다 (각각 $10^4$, $10^4$, $10^2$ 점; 첫 패널은 100 개 중 1 개만 찍는다). 세로축은 창마다 자동이다 — 창 폭이 $10^{-2}$ 배가 될 때마다 세로 범위는 $\sqrt{10^{-2}}=10^{-1}$ 배가 되고, 모양의 거칠기는 그대로다. 미분 가능한 함수라면 충분히 확대했을 때 직선(접선)이 보여야 한다.

In [ ]:
t_z, B_z = brownian_paths(1, 1_000_000, 1.0, rng)             # one path, dt = 1e-6
B_z = B_z[0]
windows = [(1.0, 100), (1e-2, 1), (1e-4, 1)]                   # (window length, plotting stride)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (w, stride) in zip(axes, windows):
    m = int(round(w * 1_000_000))
    ax.plot(t_z[:m + 1:stride], B_z[:m + 1:stride], lw=0.6, color="C0")
    ax.set_xlim(0, w); ax.set_xlabel("t")
    ax.set_title(f"window [0, {w:g}]  ({m // stride} points)")
    y_range = np.ptp(B_z[:m + 1])
    ax.text(0.03, 0.05, f"y-range {y_range:.2e} ~ sqrt({w:g}) = {np.sqrt(w):.0e}", transform=ax.transAxes, fontsize=8)
fig.suptitle("Fig 5. One Brownian path zoomed: [0,1] / [0,0.01] / [0,0.0001] — equally rough at every scale")
plt.tight_layout(); plt.show()

### 4.6 차분몫의 분산 $1/h$ (Fig 6)

$h=10^{-3}$ 격자 위 경로들에서 $D=(B_{t+h}-B_t)/h$ 를 모든 경로·시점에 대해 모아 $E[D^2]=1/h=1000$ 을 잰다. 그리고 `coarsen` 으로 **같은 경로**를 $h\in\{10^{-1},10^{-2},10^{-3}\}$ 에서 다시 보고 차분몫의 표본 표준편차를 $h$ 에 대해 log-log 로 그린다. 기준선은 $h^{-1/2}$: 창을 100 배 줄이면 기울기의 표준편차는 10 배 커진다.

In [ ]:
N_STEPS_DQ, H_LIST = 1000, [0.1, 0.01, 0.001]
t_d, B_d = brownian_paths(N_PATHS_DQ, N_STEPS_DQ, 1.0, rng)   # h = 1e-3
h = t_d[1] - t_d[0]
D = np.diff(B_d, axis=1) / h
slope_sq_hat = mc_estimate(D.ravel() ** 2)                    # N_PATHS_DQ * 1000 increments
print(f"h = {h:g}: E[D_h^2] = {slope_sq_hat}   (exact 1/h = {1/h:g}, sd(D_h) = {np.sqrt(1/h):.2f})")

sd_dq = []
for h_i in H_LIST:
    t_c, B_c = coarsen(t_d, B_d, int(round(h_i / h)))          # same paths on a coarser grid
    sd_dq.append(np.std(np.diff(B_c, axis=1) / (t_c[1] - t_c[0])))
for h_i, s in zip(H_LIST, sd_dq):
    print(f"  h = {h_i:<6g} sample sd of (B_(t+h) - B_t)/h = {s:8.3f}   vs h^(-1/2) = {h_i**-0.5:8.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
h_arr = np.array(H_LIST)
ax.loglog(h_arr, sd_dq, "o", ms=8, label="sample sd of difference quotient (same paths, coarsened)")
ax.loglog(h_arr, h_arr ** -0.5, "k--", lw=1, label="$h^{-1/2}$  (Var = 1/h)")
ax.set_xlabel("window h (log)"); ax.set_ylabel("sd of $(B_{t+h} - B_t)/h$ (log)"); ax.legend(fontsize=8)
ax.set_title("Fig 6. The difference quotient diverges like $h^{-1/2}$: no derivative")
plt.show()

점 세 개가 기울기 $-1/2$ 직선 위에 있다. $h\to0$ 에서 차분몫은 어떤 값으로도 수렴하지 않고 폭 $h^{-1/2}$ 로 퍼진다 — 정리 3 의 시뮬레이션 버전이다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드에서 계산한다: 이항 생존함수, $2(1-\Phi(1))$, 정리 1 의 (b)·(c)·(d), $1/h$.

In [ ]:
exact = {
    "p_walk400_gt1": binom.sf(210, N_STEPS_RW, 0.5),           # P(S_400 >= 22), parity!
    "p_absB1_gt1": 2 * (1 - norm.cdf(1.0)),
    "var_incr_1_3": 3.0 - 1.0,                                 # stationary increments: t - s
    "cov_incr": 0.0,                                           # independent increments
    "cov_B1_B4": min(1.0, 4.0),                                # Theorem 1(b)
    "corr_B1_B4": np.sqrt(1.0 / 4.0),                          # Theorem 1(c)
    "selfsim": 1.0,                                            # Var(c^{-1/2} B_c) = 1, c = 4
    "mean_sq_slope_h": 1.0 / h,                                # Var(D_h) = h / h^2
}
corr_hat = cov_B1B4_hat.mean / 2                               # rho = E[B_1 B_4] / sqrt(1*4); se scales by 1/2
rows = [
    {"name": "P(S_400/20 > 1)  [binomial, not N(0,1)]", "predicted": predictions["p_walk400_gt1"], "estimate": p_walk_hat, "exact": exact["p_walk400_gt1"]},
    {"name": "P(|B_1| > 1)", "predicted": predictions["p_absB1_gt1"], "estimate": p_absB1_hat, "exact": exact["p_absB1_gt1"]},
    {"name": "E[(B_3 - B_1)^2]", "predicted": predictions["var_incr_1_3"], "estimate": var_incr_hat, "exact": exact["var_incr_1_3"]},
    {"name": "E[B_1 (B_3 - B_1)]", "predicted": None, "estimate": cov_incr_hat, "exact": exact["cov_incr"]},
    {"name": "E[B_1 B_4]", "predicted": predictions["cov_B1_B4"], "estimate": cov_B1B4_hat, "exact": exact["cov_B1_B4"]},
    {"name": "corr(B_1, B_4) = E[B_1 B_4]/2", "predicted": predictions["corr_B1_B4"], "estimate": corr_hat, "se": cov_B1B4_hat.se / 2, "exact": exact["corr_B1_B4"]},
    {"name": "E[(B_4/2)^2]  (self-similarity)", "predicted": None, "estimate": selfsim_hat, "exact": exact["selfsim"]},
    {"name": "E[D_h^2], h = 1e-3", "predicted": predictions["mean_sq_slope_h"], "estimate": slope_sq_hat, "exact": exact["mean_sq_slope_h"]},
]
Markdown(compare_table(rows))

각 행이 어긋난다면 무엇이 의심되는가:
- **P(S_400/20 > 1)**: 이항값이 아니라 정규값 0.1587 에 맞는다면 걸음이 $\pm1$ 이 아니거나(격자가 없음) 부등호 규약이 다르다; 둘 다 아니면 패리티를 무시한 bin 폭처럼 "$>1$" 을 "$\ge1$" 로 세었을 가능성.
- **P(|B_1| > 1)**: 증분 분산이 $dt$ 가 아니라 $dt^2$ 나 $\sqrt{dt}$ 로 생성됐다(스케일 실수).
- **E[(B_3 - B_1)^2]**: 정상성 위반 — 열 인덱스가 시각과 어긋났다($t_{10}\ne1$).
- **E[B_1 (B_3 - B_1)]**: 독립 증분 위반 — 경로 생성에서 증분이 재사용/상관됐다.
- **E[B_1 B_4], corr**: 공분산이 $\min$ 이 아니면 (예: $st$ 면) 과정이 BM 이 아니라 $tZ$ 같은 것이다.
- **E[(B_4/2)^2]**: 자기유사 지수가 $1/2$ 가 아니다 — 증분 분산이 $t-s$ 에 비례하지 않는다.
- **E[D_h^2]**: $h$ 와 `np.diff` 축이 어긋났거나 `coarsen` 의 배수 계산이 틀렸다.

In [ ]:
assert_close(p_walk_hat, exact["p_walk400_gt1"], k=4, name="P(S_400/20>1) vs binomial")
assert_close(p_absB1_hat, exact["p_absB1_gt1"], k=4, name="P(|B_1|>1)")
assert_close(var_incr_hat, exact["var_incr_1_3"], k=4, name="E[(B_3-B_1)^2]")
assert_close(cov_incr_hat, exact["cov_incr"], k=4, name="E[B_1(B_3-B_1)]")
assert_close(cov_B1B4_hat, exact["cov_B1_B4"], k=4, name="E[B_1 B_4]")
assert_close(selfsim_hat, exact["selfsim"], k=4, name="E[(B_4/2)^2]")
assert_close(slope_sq_hat, exact["mean_sq_slope_h"], k=4, name="E[D_h^2], h=1e-3")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

손으로: (a) $\mathrm{Cov}(B_s,B_t)=\min(s,t)$ 를 증분 분해로 다시 유도하라. (b) $P(B_1>0,\ B_2>0)$ 을 구하라. 힌트: $(B_1,B_2)$ 는 상관 $\rho$ 인 이변량 정규이고 $P(X>0,Y>0)=\frac14+\frac{\arcsin\rho}{2\pi}$ (Sheppard 공식). (c) $P(B_2>0\mid B_1=1)$ 은?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $s<t$: $B_t=B_s+(B_t-B_s)$, 두 항은 독립이고 평균 0 이므로 $E[B_sB_t]=E[B_s^2]+E[B_s]E[B_t-B_s]=s+0=\min(s,t)$ (정리 1(b)).

(b) $\rho(B_1,B_2)=\mathrm{Cov}(B_1,B_2)/\sqrt{1\cdot2}=1/\sqrt2$ 이고 $\arcsin(1/\sqrt2)=\pi/4$ 이므로 $P=\frac14+\frac{\pi/4}{2\pi}=\frac14+\frac18=\frac38=0.375$.

(c) $B_2-B_1\sim N(0,1)$ 이 $B_1$ 과 독립이므로 $P(B_2>0\mid B_1=1)=P(B_2-B_1>-1)=\Phi(1)\approx0.8413$.

```python
Z1, Z2 = rng.standard_normal((2, 200_000))
b1, b2 = Z1, Z1 + Z2                                  # B_1, B_2 = B_1 + (B_2 - B_1)
print(mc_proportion((b1 > 0) & (b2 > 0)), "vs 3/8 =", 0.25 + np.arcsin(1 / np.sqrt(2)) / (2 * np.pi))
print(mc_proportion(b2[np.abs(b1 - 1) < 0.05] > 0), "vs Phi(1) =", norm.cdf(1))   # condition on B_1 ≈ 1
```

</details>

### E2 유도 후 시뮬레이션 검증

스케일된 걸음에 대해 $E[W^{(n)}_sW^{(n)}_t]=\lfloor ns\rfloor/n$ ($s\le t$) 임을 유도하라. 그리고 $n=400$, $s=1/4$, $t=1$ 에 대해 `scaled_random_walk` 로 `mc_estimate(W[:, 100] * W[:, 400])` 을 구해 `assert_close(..., 100/400, k=4)` 로 검증하라. 표준오차가 얼마일지도 먼저 예측해 보라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$k=\lfloor ns\rfloor\le m=\lfloor nt\rfloor$ 라 하자. $S_m=S_k+(S_m-S_k)$ 이고 $S_m-S_k$ 는 걸음 $X_{k+1},\dots,X_m$ 만의 함수라 $S_k$ 와 독립, 평균 0. 따라서 $E[S_kS_m]=E[S_k^2]=k$ ($E[S_k^2]=\sum_{i\le k}E[X_i^2]=k$, 교차항은 독립·평균 0). $n$ 으로 나누면 $E[W^{(n)}_sW^{(n)}_t]=k/n=\lfloor ns\rfloor/n\to s=\min(s,t)$ — BM 의 공분산이 걸음 수준에서 이미 성립한다. $n=400$, $s=1/4$: $100/400=0.25$.

표준오차: $\mathrm{Var}(W_sW_t)=E[W_s^2W_t^2]-s^2$ 이고 $W_t=W_s+\Delta$ (독립) 이므로 $E[W_s^2W_t^2]=E[W_s^4]+E[W_s^2]E[\Delta^2]\approx3s^2+s(t-s)$; 따라서 $\mathrm{Var}\approx2s^2+s(t-s)=0.125+0.1875=0.3125$, se $\approx\sqrt{0.3125/20000}\approx0.004$.

```python
_, W_e2 = scaled_random_walk(N_PATHS, 400, 1.0, rng)
cov_walk_hat = mc_estimate(W_e2[:, 100] * W_e2[:, 400])
print(cov_walk_hat, "vs floor(400/4)/400 =", 100 / 400, "| predicted se ~", np.sqrt(0.3125 / N_PATHS).round(4))
assert_close(cov_walk_hat, 100 / 400, k=4, name="E[W_(1/4) W_1]")
```

</details>

### E3 가정을 깨보기

걸음 $X_i$ 를 (a) $P(X=1)=0.55$ (평균 0.1) 로, (b) 대칭 파레토 꼬리 $P(|X|>x)=x^{-1.5}$ ($x\ge1$; `pareto_steps(size, 1.5, rng)`, 분산 무한) 로 바꾸고 4.1 의 Donsker 그림과 4.2 의 히스토그램을 다시 그려라. 무엇이 무너지는가? (b) 에서 $n=100,1000,10000$ 의 $S_n/\sqrt n$ 표본 표준편차(그리고 더 안정적인 사분위 범위 IQR)를 비교하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**(a) 평균 $\ne0$.** $S_k=0.1k+M_k$ ($M_k$ 는 평균 0 마팅게일) 이므로 $W^{(n)}_t=0.1\sqrt n\,t+M_{\lfloor nt\rfloor}/\sqrt n\approx0.1\sqrt n\,t+\sqrt{0.99}\,B_t$: 드리프트 $0.1\sqrt n\,t$ 가 $\sqrt n$ 으로 발산한다. Donsker 그림에서 $n$ 이 커질수록 경로가 직선 $0.1\sqrt n\,t$ 를 따라 위로 치솟고 $n=400$ 히스토그램의 중심은 $0.1\times20=2$ 로 이동한다. 처방: 평균을 빼고 $(S_k-0.1k)/\sqrt{n\cdot0.99}$ 로 스케일해야 CLT/Donsker 가 돌아온다.

**(b) 분산 무한.** $\alpha=1.5<2$ 이면 정규 극한이 없다. 올바른 스케일은 $n^{1/\alpha}=n^{2/3}$ 이고 극한은 점프가 있는 1.5-안정(stable) 과정이다. $S_n/\sqrt n$ 의 전형적 크기는 $n^{1/\alpha-1/2}=n^{1/6}$ 으로 자라며 ($n$ 이 10 배면 $10^{1/6}\approx1.47$ 배) Donsker 그림에서 $n$ 이 커져도 사라지지 않는 점프가 보인다. 표본 표준편차는 무한분산 때문에 가장 큰 점프 하나가 좌우하므로 실행마다 크게 달라지고 $n^{1/6}$ 법칙이 잘 안 보인다; IQR 같은 분위수 척도는 10 배 $n$ 당 약 1.5 배로 깨끗하게 자란다.

```python
# (a) biased steps: drift 0.1*sqrt(n)*t; histogram of S_400/20 centred at 2
steps_a = np.where(rng.random((N_PATHS, 400)) < 0.55, 1.0, -1.0)
S400_a = steps_a.sum(1) / 20
print("biased walk: mean of S_400/20 =", mc_estimate(S400_a), "(drift 0.1*20 = 2)")

# (b) Pareto steps, alpha = 1.5: typical size of S_n/sqrt(n) grows like n^(1/6)
for n in (100, 1000, 10_000):
    Sn = pareto_steps((2000, n), 1.5, rng).sum(1) / np.sqrt(n)
    q25, q75 = np.percentile(Sn, [25, 75])
    print(f"n={n:>6d}: sample sd = {Sn.std():7.2f} (erratic), IQR = {q75 - q25:6.2f}, n^(1/6) = {n ** (1/6):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (name, S_one) in zip(axes, [("biased p=0.55", np.cumsum(np.r_[0, np.where(rng.random(10_000) < 0.55, 1, -1)])),
                                    ("Pareto alpha=1.5", np.cumsum(np.r_[0, pareto_steps(10_000, 1.5, rng)]))]):
    for n in (100, 10_000):
        t_n, W_n = walk_rescale(S_one, n); ax.plot(t_n, W_n, drawstyle="steps-post", lw=0.8, label=f"n = {n}")
    ax.set_title(f"Donsker picture with {name} steps"); ax.set_xlabel("t"); ax.legend()
plt.tight_layout(); plt.show()
```

</details>

## 7. 정리

1. 스케일된 랜덤 워크 $S_{\lfloor nt\rfloor}/\sqrt n$ 은 $n\to\infty$ 에서 브라운 운동으로 수렴한다 (Donsker). 한 걸음을 여러 $n$ 으로 보면 모양은 다르지만 분포는 같고, 앞 패널은 다음 패널의 왼쪽 $1/10$ 을 $(10,\sqrt{10})$ 배 확대한 것이다.
2. 표준 BM = 출발점 0 + 독립 증분 + $B_t-B_s\sim N(0,t-s)$ + 연속 경로. 이 네 조건에서 $\mathrm{Cov}(B_s,B_t)=\min(s,t)$ 와 자기유사성 $c^{-1/2}B_{ct}\overset d=B_t$ 가 나온다.
3. 독립인 것은 증분이지 값이 아니다: $\rho(B_1,B_4)=1/2$, $E[B_4\mid B_1]=B_1$.
4. 차분몫 $(B_{t+h}-B_t)/h$ 의 분산은 $1/h$ — 경로는 연속이지만 어디서도 미분 불가능하다 (확대해도 직선이 안 보인다).
5. 시뮬레이션은 격자 위 근사이며 한 시점 분포는 이항(패리티!)으로 정확히 계산할 수 있다; 격자 사이의 정보는 06c 에서 다룬다.

**Trap 카드.** Q: $B_1$ 과 $B_4$ 는 독립인가? → A: 아니다. 독립인 것은 증분 $B_1$ 과 $B_4-B_1$ 이다. $\mathrm{Cov}(B_1,B_4)=\min(1,4)=1$, 상관계수 $\sqrt{1/4}=0.5$.

**다음 노트북: 06b.** 이제 '거칠음'을 정량화한다: 공분산 $\min(s,t)$ 를 가진 가우스 과정으로 BM 을 다시 만들고(Cholesky), 이차 변동이 $t$ 로 수렴하는 반면 총변동은 발산함을 확인한다 — 07a 의 이토 적분이 필요한 이유다.

`log/progress.md` 한 줄 템플릿: `- [ ] 06a 랜덤 워크→BM | 날짜: ____ | Compare 7/7 ok | 막힌 곳: ____ | 다음: 06b`